# 04 — Tyre Stint Processing
---
**Purpose:** Reconstruct tyre sequences, define globally unique `StintId`s, independently calculate `StintLap` (phase of the stint) vs `TyreLife` (physical age), and resolve telemetry dropouts.

**Input:** `outputs/cleaned_laps.parquet` (Full race sequence with flags)
**Outputs:** 
- `outputs/tyre_stints.parquet`
- `outputs/stint_quality_report.csv`
- `outputs/plots/tyre_life_distribution.png`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = os.path.join("..", "outputs")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)
input_path = os.path.join(OUTPUT_DIR, "cleaned_laps.parquet")

print("Loading dataset...")
df = pd.read_parquet(input_path)
print(f"Loaded {len(df):,} laps.")

Loading dataset...
Loaded 101,290 laps.


## 1. Sort & Forward-Fill Telemetry Dropouts

In [2]:
# Sort strictly chronologically per driver
df = df.sort_values(['Year', 'Round', 'Driver', 'LapNumber'])

# Forward-fill Stint, TyreLife, Compound within Race & Driver to bridge 1-lap dropouts
cols_to_fill = ['Stint', 'TyreLife', 'Compound']
for col in cols_to_fill:
    df[col] = df.groupby(['RaceId', 'Driver'])[col].ffill(limit=1)

print(f"Missing values remaining after ffill:")
for col in cols_to_fill:
    print(f"  {col}: {df[col].isna().sum()}")

Missing values remaining after ffill:
  Stint: 354
  TyreLife: 948
  Compound: 762


## 2. Define Unique Stints & StintLap

In [3]:
# Stint counter resets per race/driver, so we create a global ID
df['StintId'] = df['RaceId'].astype(str) + '_' + df['Driver'] + '_S' + df['Stint'].astype(str)
df.loc[df['Stint'].isna(), 'StintId'] = np.nan

# StintLap: Sequential counter within the StintId (phase of the stint)
df['StintLap'] = df.groupby('StintId').cumcount() + 1
df.loc[df['StintId'].isna(), 'StintLap'] = np.nan

# Preserve original TyreLife, but also assign it to TyreAge as requested
df['TyreAge'] = df['TyreLife']

# TyreChangeFlag
df['PrevStintId'] = df.groupby(['RaceId', 'Driver'])['StintId'].shift(1)
df['TyreChangeFlag'] = (df['StintId'] != df['PrevStintId']) & df['PrevStintId'].notna() & df['StintId'].notna()
df = df.drop(columns=['PrevStintId'])

print("Globally unique StintId, StintLap, TyreAge, and TyreChangeFlag created.")

Globally unique StintId, StintLap, TyreAge, and TyreChangeFlag created.


## 3. Stint Quality Flagging

In [4]:
# A stint needs at least 3 valid racing laps to establish a baseline.
clean_laps_per_stint = df.groupby('StintId')['is_clean_lap'].transform('sum')
# 1 = Good Stint, 0 = Bad Stint (e.g., short stints < 3 clean laps)
df['StintQualityFlag'] = np.where(clean_laps_per_stint >= 3, 1, 0)
df.loc[df['StintId'].isna(), 'StintQualityFlag'] = 0

good_stints = df[df['StintQualityFlag'] == 1]['StintId'].nunique()
bad_stints = df[df['StintQualityFlag'] == 0]['StintId'].nunique() - 1 # exclude NaN
print(f"Identified {good_stints:,} high-quality stints and {bad_stints:,} short/low-quality stints.")

Identified 4,138 high-quality stints and 1,044 short/low-quality stints.


## 4. Rigorous Data Quality Validation

In [5]:
# 1. Unique Compound per Stint
compound_check = df.groupby('StintId')['Compound'].nunique()
if compound_check.max() > 1:
    print("[WARNING] Found a StintId containing multiple compounds!")
else:
    print("[OK] All StintIds contain exactly 1 compound.")

# 2. StintLap starts at 1
min_stintlap = df.dropna(subset=['StintId']).groupby('StintId')['StintLap'].min()
if min_stintlap.max() > 1:
    print("[WARNING] StintLap does not start at 1 for all stints!")
else:
    print("[OK] StintLap correctly starts at 1.")

# 3. TyreLife Monotonicity
tyrelife_diff = df.groupby('StintId')['TyreLife'].diff()
non_monotonic = df[tyrelife_diff < 0]
if len(non_monotonic) > 0:
    print(f"[WARNING] Found {len(non_monotonic)} laps where TyreLife decreased within a stint!")
else:
    print("[OK] TyreLife is perfectly monotonic within all stints.")

[OK] All StintIds contain exactly 1 compound.
[OK] StintLap correctly starts at 1.
[OK] TyreLife is perfectly monotonic within all stints.


## 5. Diagnostic Reporting (Quality by Dimension)

In [6]:
report_data = []
def report_dimension(dim_col):
    stint_level = df.dropna(subset=['StintId']).groupby([dim_col, 'StintId'])['StintQualityFlag'].first().reset_index()
    res = stint_level.groupby(dim_col).agg(
        Total_Stints=('StintId', 'count'),
        Good_Stints=('StintQualityFlag', 'sum')
    )
    res['Quality_Pct'] = (res['Good_Stints'] / res['Total_Stints']) * 100
    return res

dimensions = ['Year', 'GrandPrix', 'CanonicalTeam', 'Compound']

for dim in dimensions:
    if dim in df.columns:
        res = report_dimension(dim)
        print(f"\n--- Stint Quality by {dim} ---")
        print(res.sort_values('Quality_Pct', ascending=False).head(10))
        for idx, row in res.iterrows():
            report_data.append({
                'Dimension': dim,
                'Value': str(idx),
                'Total_Stints': row['Total_Stints'],
                'Good_Stints': row['Good_Stints'],
                'Quality_Pct': row['Quality_Pct']
            })

report_df = pd.DataFrame(report_data)
report_out = os.path.join(OUTPUT_DIR, "stint_quality_report.csv")
report_df.to_csv(report_out, index=False)
print(f"\nSaved quality report to {report_out}")


--- Stint Quality by Year ---
      Total_Stints  Good_Stints  Quality_Pct
Year                                        
2024          1300         1054    81.076923
2023          1348         1072    79.525223
2025          1295         1029    79.459459
2022          1240          983    79.274194



--- Stint Quality by GrandPrix ---
                          Total_Stints  Good_Stints  Quality_Pct
GrandPrix                                                       
Spanish Grand Prix                 271          267    98.523985
French Grand Prix                   45           44    97.777778
Bahrain Grand Prix                 273          266    97.435897
United States Grand Prix           196          189    96.428571
Abu Dhabi Grand Prix               203          195    96.059113
Miami Grand Prix                   171          164    95.906433
Chinese Grand Prix                 105          100    95.238095
Hungarian Grand Prix               227          216    95.154185
Saudi Arabian Grand Prix           159          149    93.710692
Italian Grand Prix                 180          167    92.777778

--- Stint Quality by CanonicalTeam ---
               Total_Stints  Good_Stints  Quality_Pct
CanonicalTeam                                        
Alpine                  495         


--- Stint Quality by Compound ---
              Total_Stints  Good_Stints  Quality_Pct
Compound                                            
HARD                  1752         1592    90.867580
MEDIUM                1997         1801    90.185278
SOFT                   959          745    77.685089
INTERMEDIATE           424            0     0.000000
WET                     51            0     0.000000

Saved quality report to ..\outputs\stint_quality_report.csv


## 6. Diagnostic Plots

In [7]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df[df['StintQualityFlag']==1], x='TyreLife', hue='Compound', bins=30, multiple='stack')
plt.title('TyreLife Distribution in High-Quality Stints')
plt.xlabel('Physical Tyre Age (Laps)')
plt.ylabel('Lap Count')
plot_path = os.path.join(PLOT_DIR, 'tyre_life_distribution.png')
plt.savefig(plot_path)
plt.close()
print(f"Saved diagnostic plot to {plot_path}")

Saved diagnostic plot to ..\outputs\plots\tyre_life_distribution.png


## 7. Save Processed Stints

In [8]:
out_path = os.path.join(OUTPUT_DIR, "tyre_stints.parquet")
df.to_parquet(out_path, index=False)
fsize = os.path.getsize(out_path) / 1e6
print(f"Saved stint dataset to: {out_path}")
print(f"File size: {fsize:.1f} MB")
print("\n[OK] Notebook 04 Tyre Stint Processing complete.")

Saved stint dataset to: ..\outputs\tyre_stints.parquet
File size: 6.7 MB

[OK] Notebook 04 Tyre Stint Processing complete.
